# 🏠 Dự đoán Giá Nhà — Buổi 7/9
## Model Evaluation: Đánh Giá Model Toàn Diện

---

### 🗺️ Lộ trình 9 buổi

| # | Chủ đề | Trạng thái |
|---|--------|-----------|
| 1 | Giới thiệu dự án & lộ trình | ✅ Hoàn thành |
| 2 | Setup môi trường & khám phá SalePrice | ✅ Hoàn thành |
| 3 | EDA — Phân tích missing, tương quan, outliers | ✅ Hoàn thành |
| 4 | Data Cleaning & Preprocessing | ✅ Hoàn thành |
| 5 | Feature Engineering | ✅ Hoàn thành |
| 6 | Model Training (Linear + Tree + Hyperparameters + Stacking) | ✅ Hoàn thành |
| **7** | **Model Evaluation — Đánh Giá Toàn Diện** | 🔄 **Buổi này** |
| 8 | Kaggle Submission & Cải Thiện Kết Quả | ⏳ Tiếp theo |
| 9 | Quiz Tốt Nghiệp | ⏳ |

---

### 🎯 Mục tiêu buổi 7

Sau buổi này bạn sẽ biết:
1. **Metrics:** RMSE, MAE, R² — đọc và hiểu từng chỉ số
2. **Cross-Validation:** Phân tích RMSE từng fold để hiểu độ ổn định
3. **Learning Curves:** Phát hiện model đang bị overfit hay underfit
4. **Residual Analysis:** Xem model còn sai ở đâu
5. **So sánh tổng quát:** Biểu đồ + bảng đánh giá tất cả models

> 💡 **Lưu ý:** Buổi này **không train model mới** — ta chỉ đánh giá và phân tích sâu hơn các models đã train ở buổi 6!


In [ ]:
# ============================================================
# ♻️ SETUP — Import thư viện & rebuild pipeline từ Buổi 5
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# --- Linear models ---
from sklearn.linear_model import RidgeCV, LassoCV, Ridge

# --- Tree-based models ---
from sklearn.ensemble import RandomForestRegressor, StackingRegressor

# --- XGBoost (cần cài: pip install xgboost) ---
try:
    from xgboost import XGBRegressor
    HAVE_XGB = True
    print("✅ XGBoost sẵn sàng")
except ImportError:
    HAVE_XGB = False
    print("⚠️  XGBoost chưa cài — cài bằng: pip install xgboost")

# --- Đánh giá ---
from sklearn.model_selection import cross_val_score, KFold, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pd.set_option('display.float_format', lambda x: f'{x:.4f}')

# ── Load dữ liệu ──────────────────────────────────────────
train_df = pd.read_csv('data/train.csv')
test_df  = pd.read_csv('data/test.csv')

train_id = train_df['Id'].copy()
test_id  = test_df['Id'].copy()
y = np.log1p(train_df['SalePrice'])

train_df.drop(['Id', 'SalePrice'], axis=1, inplace=True)
test_df.drop(['Id'], axis=1, inplace=True)

ntrain = len(train_df)
ntest  = len(test_df)
all_data = pd.concat([train_df, test_df], axis=0, ignore_index=True)

# ── Fill missing ──────────────────────────────────────────
none_cols = ['PoolQC','MiscFeature','Alley','Fence','FireplaceQu',
             'GarageType','GarageFinish','GarageQual','GarageCond',
             'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','MasVnrType']
zero_cols = ['GarageYrBlt','GarageArea','GarageCars','MasVnrArea',
             'BsmtFinSF1','BsmtFinSF2','BsmtUnfSF','TotalBsmtSF','BsmtFullBath','BsmtHalfBath']
mode_cols = ['MSZoning','Utilities','Functional','Electrical',
             'KitchenQual','Exterior1st','Exterior2nd','SaleType']

for col in none_cols: all_data[col] = all_data[col].fillna('None')
for col in zero_cols: all_data[col] = all_data[col].fillna(0)
all_data['LotFrontage'] = (all_data.groupby('Neighborhood')['LotFrontage']
                           .transform(lambda x: x.fillna(x.median())))
for col in mode_cols: all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

# ── Xoá outlier ──────────────────────────────────────────
outlier_mask = (all_data['GrLivArea'][:ntrain] > 4000) & (np.expm1(y) < 300_000)
outlier_idx  = outlier_mask[outlier_mask].index.tolist()
all_data.drop(index=outlier_idx, inplace=True)
all_data.reset_index(drop=True, inplace=True)
y.drop(index=outlier_idx, inplace=True)
y.reset_index(drop=True, inplace=True)
ntrain = ntrain - len(outlier_idx)

# ── Feature Engineering ───────────────────────────────────
all_data['TotalSF']      = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']
all_data['TotalPorchSF'] = (all_data['OpenPorchSF'] + all_data['3SsnPorch'] +
                             all_data['EnclosedPorch'] + all_data['ScreenPorch'] + all_data['WoodDeckSF'])
all_data['TotalBath']    = (all_data['FullBath'] + 0.5*all_data['HalfBath'] +
                             all_data['BsmtFullBath'] + 0.5*all_data['BsmtHalfBath'])
all_data['OverallScore'] = all_data['OverallQual'] * all_data['OverallCond']
all_data['HasGarage']    = (all_data['GarageArea'] > 0).astype(int)
all_data['HasBsmt']      = (all_data['TotalBsmtSF'] > 0).astype(int)
all_data['HouseAge']     = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodAge']     = all_data['YrSold'] - all_data['YearRemodAdd']

for feat in ['OverallQual', 'GrLivArea', 'TotalSF']:
    all_data[f'{feat}_sq']   = all_data[feat] ** 2
    all_data[f'{feat}_sqrt'] = np.sqrt(all_data[feat])

for feat in ['GrLivArea', 'TotalSF', 'LotArea']:
    all_data[f'{feat}_log'] = np.log1p(all_data[feat])

# ── Ordinal Encoding ──────────────────────────────────────
ordinal_mappings = {
    'ExterQual':    ['None','Po','Fa','TA','Gd','Ex'],
    'ExterCond':    ['None','Po','Fa','TA','Gd','Ex'],
    'BsmtQual':     ['None','Po','Fa','TA','Gd','Ex'],
    'BsmtCond':     ['None','Po','Fa','TA','Gd','Ex'],
    'BsmtExposure': ['None','No','Mn','Av','Gd'],
    'BsmtFinType1': ['None','Unf','LwQ','Rec','BLQ','ALQ','GLQ'],
    'BsmtFinType2': ['None','Unf','LwQ','Rec','BLQ','ALQ','GLQ'],
    'HeatingQC':    ['None','Po','Fa','TA','Gd','Ex'],
    'KitchenQual':  ['None','Po','Fa','TA','Gd','Ex'],
    'FireplaceQu':  ['None','Po','Fa','TA','Gd','Ex'],
    'GarageFinish': ['None','Unf','RFn','Fin'],
    'GarageQual':   ['None','Po','Fa','TA','Gd','Ex'],
    'GarageCond':   ['None','Po','Fa','TA','Gd','Ex'],
    'PoolQC':       ['None','Fa','TA','Gd','Ex'],
    'Fence':        ['None','MnWw','GdWo','MnPrv','GdPrv'],
    'Functional':   ['Sal','Sev','Maj2','Maj1','Mod','Min2','Min1','Typ'],
    'LandSlope':    ['Sev','Mod','Gtl'],
    'LotShape':     ['IR3','IR2','IR1','Reg'],
    'PavedDrive':   ['N','P','Y'],
}
for col, order in ordinal_mappings.items():
    mapping = {val: i for i, val in enumerate(order)}
    all_data[col] = all_data[col].map(mapping).fillna(0).astype(int)

# ── One-Hot + Variance Filter + Scale ────────────────────
cat_cols     = all_data.select_dtypes(include=['object']).columns.tolist()
all_data_enc = pd.get_dummies(all_data, columns=cat_cols, drop_first=True)

X_train_raw = all_data_enc[:ntrain].copy()
X_test_raw  = all_data_enc[ntrain:].copy()
X_test_raw.reset_index(drop=True, inplace=True)

selector      = VarianceThreshold(threshold=0.01)
selected_mask = selector.fit(X_train_raw).get_support()
X_train_sel   = X_train_raw.loc[:, selected_mask]
X_test_sel    = X_test_raw.loc[:, selected_mask]

scaler  = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train_sel),
                        columns=X_train_sel.columns)
X_test  = pd.DataFrame(scaler.transform(X_test_sel),
                        columns=X_test_sel.columns)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

def rmse_cv(model):
    scores = cross_val_score(model, X_train, y,
                             scoring='neg_root_mean_squared_error', cv=kf)
    return -scores.mean()

# ── Train các model để dùng trong buổi này ───────────────
print("Training models (chờ vài giây)...")

ridge = RidgeCV(alphas=[1, 10, 50, 100, 300])
ridge.fit(X_train, y)
ridge_rmse = rmse_cv(ridge)

lasso = LassoCV(alphas=[0.0001, 0.001, 0.01], cv=kf, max_iter=5000)
lasso.fit(X_train, y)
lasso_rmse = rmse_cv(lasso)

rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y)
rf_rmse = rmse_cv(rf)

if HAVE_XGB:
    xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4,
                        subsample=0.8, random_state=42, verbosity=0)
    xgb.fit(X_train, y)
    xgb_rmse = rmse_cv(xgb)

# Model tốt nhất để dùng trong phân tích
best_model      = xgb if HAVE_XGB else rf
best_model_name = "XGBoost" if HAVE_XGB else "Random Forest"

print("✅ Sẵn sàng!")
print(f"   Ridge         RMSE = {ridge_rmse:.4f}")
print(f"   Lasso         RMSE = {lasso_rmse:.4f}")
print(f"   Random Forest RMSE = {rf_rmse:.4f}")
if HAVE_XGB:
    print(f"   XGBoost       RMSE = {xgb_rmse:.4f}")
print(f"\n   → Model tốt nhất dùng để phân tích: {best_model_name}")

**Expected Output:**

![img7-1](images/img_buoi7/img7-1.png)

> ⬆️ **Chạy cell Setup trước** — toàn bộ pipeline và models sẽ được chuẩn bị tự động.

### 💡 Setup đã làm gì?

Cell trên tái tạo lại toàn bộ pipeline từ buổi 4–6:
1. **Load data** → train.csv + test.csv
2. **Fill missing** → None/0/median/mode theo từng loại cột
3. **Feature Engineering** → TotalSF, HouseAge, log/polynomial features
4. **Ordinal Encoding** → chuyển text có thứ tự thành số
5. **One-Hot Encoding** → chuyển text không có thứ tự thành cột binary
6. **Variance Filter + Scale** → loại features ít biến động, chuẩn hóa về cùng thang đo
7. **Train 4 models** → Ridge, Lasso, Random Forest, XGBoost

> Ở buổi 7 ta **không train thêm**, chỉ **đánh giá** các models đã có!


---

## 📏 Task 1 — Các Chỉ Số Đánh Giá: RMSE, MAE, R²

Khi model đưa ra dự đoán, ta cần **đo lường sai số** để biết model tốt đến đâu.

### Ba chỉ số phổ biến nhất

#### RMSE — Root Mean Squared Error
$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

- Phạt nặng hơn với **lỗi lớn** (vì bình phương)
- Dùng cho bài toán khi lỗi lớn nguy hiểm hơn lỗi nhỏ

#### MAE — Mean Absolute Error
$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$$

- Dễ hiểu hơn: "trung bình mỗi dự đoán sai bao nhiêu"
- Không phạt nặng outliers

#### R² — Coefficient of Determination
$$R^2 = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$

- R² = 1.0 → dự đoán hoàn hảo
- R² = 0.0 → model không tốt hơn chỉ đơn giản là đoán trung bình
- Thường thấy R² ~ 0.87–0.92 là tốt cho bài này

> **Lưu ý:** Vì y đã được log-transform, RMSE/MAE tính trên log scale. RMSE=0.12 ≈ sai ~12% theo thang log.

In [ ]:
# ── Task 1: Tính metrics cho từng model ───────────────────
def get_metrics(model, name):
    """Tính RMSE, MAE, R² trên training data."""
    y_pred     = model.predict(X_train)
    rmse_train = float(np.sqrt(mean_squared_error(y, y_pred)))
    mae_train  = float(mean_absolute_error(y, y_pred))
    r2_train   = float(r2_score(y, y_pred))
    return {'Model': name, 'Train RMSE': rmse_train, 'MAE (train)': mae_train, 'R² (train)': r2_train}

rows = [
    get_metrics(ridge, 'Ridge'),
    get_metrics(lasso, 'Lasso'),
    get_metrics(rf,    'Random Forest'),
]
if HAVE_XGB:
    rows.append(get_metrics(xgb, 'XGBoost'))

metrics_df = pd.DataFrame(rows).set_index('Model')

# Gắn CV RMSE (đã tính ở phần setup)
cv_rmse_dict = {'Ridge': ridge_rmse, 'Lasso': lasso_rmse, 'Random Forest': rf_rmse}
if HAVE_XGB:
    cv_rmse_dict['XGBoost'] = xgb_rmse
metrics_df['CV RMSE'] = [cv_rmse_dict[m] for m in metrics_df.index]
metrics_df['Overfit Gap'] = metrics_df['CV RMSE'] - metrics_df['Train RMSE']

print("📊 Metrics tổng hợp (Train vs CV):")
print(metrics_df[['Train RMSE', 'CV RMSE', 'Overfit Gap', 'R² (train)']].to_string())

# ── Biểu đồ grouped bar: Train RMSE vs CV RMSE ────────────
model_list  = list(metrics_df.index)
train_rmses = metrics_df['Train RMSE'].values
cv_rmses    = metrics_df['CV RMSE'].values
r2_vals     = metrics_df['R² (train)'].values
x = np.arange(len(model_list))
w = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: Train vs CV RMSE grouped bar ───────────────────
axes[0].bar(x - w/2, train_rmses, w, label='Train RMSE', color='#3498db', alpha=0.85, edgecolor='none')
axes[0].bar(x + w/2, cv_rmses,   w, label='CV RMSE',    color='#e74c3c', alpha=0.85, edgecolor='none')
for i, (tr, cv) in enumerate(zip(train_rmses, cv_rmses)):
    axes[0].text(i - w/2, tr + 0.001, f'{tr:.3f}', ha='center', fontsize=8)
    axes[0].text(i + w/2, cv + 0.001, f'{cv:.3f}', ha='center', fontsize=8, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_list, rotation=10, fontsize=10)
axes[0].set_ylabel('RMSE')
axes[0].set_title('Train RMSE vs CV RMSE\n(khoảng cách lớn = đang overfit)',
                   fontweight='bold')
axes[0].legend()
axes[0].grid(True, axis='y', alpha=0.3)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# ── Right: R² horizontal bar ──────────────────────────────
r2_colors = ['#27ae60' if r == max(r2_vals) else '#95a5a6' for r in r2_vals]
axes[1].barh(model_list, r2_vals, color=r2_colors, edgecolor='none', alpha=0.85)
for i, r in enumerate(r2_vals):
    axes[1].text(r - 0.005, i, f'{r:.4f}', va='center', ha='right',
                 fontsize=9, color='white', fontweight='bold')
axes[1].set_xlabel('R² (train) — cao hơn = giải thích được nhiều variance hơn')
axes[1].set_title('R² Score trên Training Data\n(tree models overfit → R² gần 1.0)',
                   fontweight='bold')
axes[1].set_xlim(min(r2_vals) * 0.97, 1.02)
axes[1].grid(True, axis='x', alpha=0.3)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle('Task 1 — Metrics: Train vs Validation Performance',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n⚠️  Tree models: Train RMSE rất thấp + R² gần 1 → đang overfit training data!")
print("   → Dùng CV RMSE (cột đỏ) để so sánh models trung thực hơn.")


**Expected Output:**

![img7-2](images/img_buoi7/img7-2.png)
![img7-3](images/img_buoi7/img7-3.png)

### 💡 Giải thích kết quả Metrics

**Đọc bảng Train Metrics:**
- Tree models (RF, XGB) có **Train RMSE rất thấp** vì chúng "học vẹt" training data — đây là dấu hiệu **overfit**!
- Linear models (Ridge, Lasso) train và CV RMSE gần nhau hơn → ổn định hơn

**Khi nào dùng chỉ số nào?**

| Tình huống | Nên dùng |
|-----------|---------|
| So sánh models với nhau | **CV RMSE** — trung thực nhất |
| Giải thích cho người không kỹ thuật | **MAE** — dễ hiểu ("sai trung bình X đồng") |
| Muốn biết model giải thích bao nhiêu % variance | **R²** |
| Submit Kaggle (bài này) | **RMSE trên log SalePrice** |

> ⚠️ **Quy tắc vàng:** Luôn dùng **CV RMSE** (không phải train RMSE) khi so sánh models! Train RMSE luôn thấp hơn vì model đã "nhìn thấy" data đó rồi.


---

## 🔁 Task 2 — Cross-Validation: Phân Tích Từng Fold

### Tại sao cần xem từng fold?

Nếu chỉ xem **trung bình CV RMSE**, ta có thể bỏ sót vấn đề:

```
Fold 1: RMSE = 0.110  ← tốt
Fold 2: RMSE = 0.112  ← tốt
Fold 3: RMSE = 0.180  ← đột nhiên xấu! Có gì đó sai?
Fold 4: RMSE = 0.111  ← tốt
Fold 5: RMSE = 0.108  ← tốt
Trung bình = 0.124  ← che giấu vấn đề!
```

Xem từng fold giúp ta biết model có **ổn định** không hay bị ảnh hưởng bởi 1 phần data.

> **Cross-Validation hoạt động thế nào?**  
> Data được chia thành 5 phần bằng nhau (fold). Mỗi lần, 4 phần dùng để train, 1 phần để kiểm tra. Quá trình lặp lại 5 lần → ta có 5 RMSE độc lập → lấy trung bình.


In [ ]:
# ── Task 2: CV fold analysis — best model + all models ────
# Lấy CV RMSE từng fold cho tất cả models
all_cv_scores = {}
for model, name in [(ridge, 'Ridge'), (lasso, 'Lasso'), (rf, 'Random Forest')]:
    sc = cross_val_score(model, X_train, y,
                         scoring='neg_root_mean_squared_error', cv=kf)
    all_cv_scores[name] = -sc
if HAVE_XGB:
    sc = cross_val_score(xgb, X_train, y,
                         scoring='neg_root_mean_squared_error', cv=kf)
    all_cv_scores['XGBoost'] = -sc

cv_best   = all_cv_scores[best_model_name]
fold_labels = [f'Fold {i}' for i in range(1, 6)]

print(f"CV RMSE từng fold — {best_model_name}:")
for i, s in enumerate(cv_best, 1):
    print(f"  Fold {i}: {s:.4f}")
print(f"\n  Trung bình : {cv_best.mean():.4f}")
print(f"  Độ lệch    : {cv_best.std():.4f}  (nhỏ = model ổn định)")

# ── 2 biểu đồ cạnh nhau ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: Bar từng fold (best model) ─────────────────────
bar_colors_fold = [
    '#e74c3c' if s > cv_best.mean() + cv_best.std() else '#3498db'
    for s in cv_best
]
bars = axes[0].bar(fold_labels, cv_best, color=bar_colors_fold,
                   width=0.55, edgecolor='none')
for bar, val in zip(bars, cv_best):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0003,
                 f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')
axes[0].axhline(cv_best.mean(), color='red', linestyle='--', linewidth=1.5,
                label=f'Mean = {cv_best.mean():.4f}\nStd = ±{cv_best.std():.4f}')
axes[0].set_title(f'CV RMSE từng Fold — {best_model_name}',
                   fontsize=11, fontweight='bold')
axes[0].set_ylabel('RMSE')
axes[0].set_ylim(cv_best.min() * 0.97, cv_best.max() * 1.04)
axes[0].legend(fontsize=9)
axes[0].grid(True, axis='y', alpha=0.3)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# ── Right: Box plot — tất cả models ─────────────────────
model_names_box = list(all_cv_scores.keys())
data_box = [all_cv_scores[n] for n in model_names_box]
box_palette = ['#3498db', '#e67e22', '#27ae60', '#9b59b6']

bp = axes[1].boxplot(data_box, patch_artist=True, notch=False,
                     medianprops=dict(color='red', linewidth=2),
                     whiskerprops=dict(linewidth=1.5),
                     capprops=dict(linewidth=1.5),
                     flierprops=dict(marker='o', markersize=5, alpha=0.5))
for patch, color in zip(bp['boxes'], box_palette[:len(model_names_box)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

axes[1].set_xticklabels(model_names_box, fontsize=10)
axes[1].set_ylabel('CV RMSE')
axes[1].set_title('Phân Phối CV RMSE — Tất Cả Models\n(hộp nhỏ = ổn định, thấp = tốt)',
                   fontsize=11, fontweight='bold')
axes[1].grid(True, axis='y', alpha=0.3)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle('Task 2 — Cross-Validation: Stability & Performance của Từng Model',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTóm tắt CV tất cả models:")
for name, scores in all_cv_scores.items():
    print(f"  {name:<15}: mean={scores.mean():.4f}  std=±{scores.std():.4f}")


**Expected Output:**

![img7-4](images/img_buoi7/img7-4.png)
![img7-5](images/img_buoi7/img7-5.png)

### 💡 Đọc kết quả CV fold

| Tình huống | Ý nghĩa | Hành động |
|-----------|---------|-----------|
| Độ lệch thấp (< 0.005) | Model ổn định — tin tưởng được | Tiếp tục dùng model này |
| Độ lệch cao (> 0.015) | Model nhạy cảm với data | Thêm regularization hoặc thêm data |
| 1 fold xấu hơn hẳn (đỏ) | Có outliers hoặc pattern đặc biệt trong fold đó | Kiểm tra data trong fold đó |

> **Quy tắc:** Model tốt không chỉ có RMSE thấp mà còn phải có **độ lệch thấp** giữa các fold!


---

## 📈 Task 3 — Learning Curves: Overfit hay Underfit?

### Bias vs. Variance — hai vấn đề phổ biến nhất

```
UNDERFIT (High Bias)          OVERFIT (High Variance)
Model quá đơn giản            Model quá phức tạp

Train RMSE: cao               Train RMSE: rất thấp
Val   RMSE: cao               Val   RMSE: cao hơn nhiều
```

### Learning Curve là gì?

Ta train model với **số lượng data tăng dần** và xem Train RMSE và Validation RMSE thay đổi thế nào:

- **Underfit:** Cả hai đều cao và gần nhau
- **Overfit:** Train thấp, Validation cao — khoảng cách lớn
- **Tốt:** Cả hai thấp và gần nhau

In [ ]:
# ── Task 3: Learning Curves ────────────────────────────────
train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_train, y,
    train_sizes = np.linspace(0.2, 1.0, 6),   # 20%, 36%, ..., 100% của data
    scoring     = 'neg_root_mean_squared_error',
    cv          = 5,
    n_jobs      = -1
)

train_rmse = -train_scores.mean(axis=1)
val_rmse   = -val_scores.mean(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_rmse, 'o-', color='#3498db', label='Train RMSE')
plt.plot(train_sizes, val_rmse,   'o-', color='#e74c3c', label='Validation RMSE')
plt.fill_between(train_sizes, train_rmse, val_rmse, alpha=0.1, color='gray')
plt.xlabel('Số lượng training samples')
plt.ylabel('RMSE')
plt.title(f'Learning Curves — {best_model_name}', fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()

gap = val_rmse[-1] - train_rmse[-1]
print(f"Khoảng cách Train vs Val RMSE (khi dùng toàn bộ data): {gap:.4f}")
if gap > 0.02:
    print("→ Model đang bị OVERFIT — thử giảm độ phức tạp hoặc thêm regularization")
elif gap < 0.005:
    print("→ Model có thể UNDERFIT — thử tăng độ phức tạp")
else:
    print("→ Model khá cân bằng ✅")

**Expected Output:**

![img7-6](images/img_buoi7/img7-6.png)
![img7-7](images/img_buoi7/img7-7.png)

### 💡 Đọc Learning Curves

| Hình dạng | Chẩn đoán | Giải pháp |
|-----------|-----------|-----------|
| Cả hai đường **cao** & gần nhau | Underfit (High Bias) | Dùng model phức tạp hơn, thêm features |
| Train **thấp**, Val **cao** (khoảng cách lớn) | Overfit (High Variance) | Thêm data, regularize, giảm complexity |
| Cả hai **thấp** & gần nhau | Cân bằng tốt ✅ | Tiếp tục, thêm data sẽ giúp thêm |
| Val **plateau** (không giảm thêm) | Đã đến giới hạn model | Đổi features hoặc thử model mới |

**Nhìn vào `gap = val_rmse[-1] - train_rmse[-1]`:**
- `gap < 0.005` → model có thể underfit
- `0.005 ≤ gap ≤ 0.02` → cân bằng tốt ✅
- `gap > 0.02` → đang overfit, cần regularize

> **Thực tế:** XGBoost thường có khoảng cách lớn hơn Ridge/Lasso — nhưng CV RMSE tốt hơn nên vẫn được chọn!


---

## 🔎 Task 4 — Residual Analysis: Model Còn Sai Ở Đâu?

**Residual** = Giá trị thực tế − Dự đoán = $y_i - \hat{y}_i$

### Model tốt thì residuals phải:
1. **Phân phối ngẫu nhiên** quanh 0 — không có pattern
2. **Phân phối chuẩn** (hình chuông) — không lệch một bên
3. **Độc lập với giá trị dự đoán** — sai số không tăng theo predicted

Nếu thấy pattern trong residuals → model chưa nắm bắt được một mối quan hệ nào đó trong data!

> **Ví dụ:** Nếu model luôn dự đoán thấp hơn với nhà đắt tiền → histogram sẽ lệch âm ở vùng giá cao → cần thêm feature hoặc dùng model phức tạp hơn.


In [ ]:
# ── Task 4: Residual plots (3 biểu đồ) ────────────────────
from scipy import stats as sp_stats

y_pred    = best_model.predict(X_train)
residuals = y - y_pred

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ── Plot 1: Scatter Residuals vs Predicted ────────────────
axes[0].scatter(y_pred, residuals, alpha=0.3, s=12, color='#3498db')
axes[0].axhline(0, color='#e74c3c', linestyle='--', linewidth=1.5)
# Trend line
z = np.polyfit(y_pred, residuals, 1)
x_line = np.linspace(y_pred.min(), y_pred.max(), 100)
axes[0].plot(x_line, np.poly1d(z)(x_line), color='#e67e22', lw=1.5, alpha=0.7, label='Xu hướng')
axes[0].set_xlabel('Predicted (log SalePrice)')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Predicted\n(điểm rải đều quanh 0 = tốt)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.2)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# ── Plot 2: Histogram + đường cong chuẩn ─────────────────
axes[1].hist(residuals, bins=40, color='#2ecc71', edgecolor='none',
             alpha=0.8, density=True, label='Residuals')
x_curve = np.linspace(residuals.min(), residuals.max(), 100)
axes[1].plot(x_curve,
             sp_stats.norm.pdf(x_curve, residuals.mean(), residuals.std()),
             'r-', lw=2, label='Phân phối chuẩn')
axes[1].axvline(0, color='red', linestyle='--', lw=1.5, alpha=0.5)
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Mật độ')
axes[1].set_title('Phân phối Residuals\n(hình chuông đối xứng = tốt)')
axes[1].legend(fontsize=9)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

# ── Plot 3: Q-Q plot ──────────────────────────────────────
(osm, osr), (slope, intercept, r) = sp_stats.probplot(residuals, dist='norm')
axes[2].scatter(osm, osr, alpha=0.4, s=12, color='#9b59b6')
axes[2].plot([min(osm), max(osm)],
             [slope * min(osm) + intercept, slope * max(osm) + intercept],
             'r--', lw=1.5, label=f'R = {r:.4f}')
axes[2].set_xlabel('Quantiles lý thuyết (Normal)')
axes[2].set_ylabel('Quantiles thực tế (Residuals)')
axes[2].set_title('Q-Q Plot\n(điểm sát đường đỏ = residuals phân phối chuẩn)')
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.2)
axes[2].spines['top'].set_visible(False)
axes[2].spines['right'].set_visible(False)

plt.suptitle(f'Residual Analysis — {best_model_name}', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Residuals — {best_model_name}:")
print(f"  Trung bình : {residuals.mean():.4f}  (gần 0 = tốt)")
print(f"  Độ lệch    : {residuals.std():.4f}")
print(f"  Skewness   : {float(pd.Series(residuals).skew()):.4f}  (gần 0 = cân bằng)")
print(f"  Min / Max  : {residuals.min():.4f} / {residuals.max():.4f}")


**Expected Output:**

![img7-8](images/img_buoi7/img7-8.png)
![img7-9](images/img_buoi7/img7-9.png)

### 💡 Đọc Residual Plots

**Plot 1 — Residuals vs Predicted:**
- ✅ Tốt: Các điểm rải đều quanh đường đỏ y = 0, không có pattern rõ ràng
- ❌ Hình phễu (variance tăng theo predicted): model dự đoán kém hơn ở nhà đắt tiền
- ❌ Hình cung/chữ U: model bỏ sót quan hệ phi tuyến nào đó

**Plot 2 — Histogram Residuals:**
- ✅ Tốt: Hình chuông đối xứng quanh 0
- ❌ Lệch trái: model đang đoán cao hơn thực tế (over-estimate)
- ❌ Lệch phải: model đang đoán thấp hơn thực tế (under-estimate)

**Plot 3 — Q-Q Plot:**
- ✅ Tốt: Đoạn giữa - Rất khớp (mô hình dự đoán cực chuẩn cho những căn nhà có giá trung bình).
- ❌ Hai đầu (đặc biệt là bên trái): Các điểm bị cong xuống và rời xa đường đỏ.
- Ý nghĩa: Mô hình đang bị "đuối" khi dự đoán những căn nhà có giá quá thấp hoặc quá cao (những trường hợp đặc biệt). Chỉ số là rất cao, chứng tỏ mô hình đạt yêu cầu về mặt thống kê.

> **Mẹo thực tế:** Nếu thấy model hay sai với một nhóm nhà cụ thể (ví dụ: nhà rất đắt hoặc rất rẻ), hãy xem lại features hoặc thêm dữ liệu cho nhóm đó.


---

## 🏆 Task 5 — So Sánh Tất Cả Models

Cuối buổi, ta tập hợp kết quả của tất cả models vào 1 bảng và 1 biểu đồ để có cái nhìn tổng quan — từ đó **chọn model tốt nhất** để tạo file submission ở buổi 8.


In [ ]:
# ── Task 5: Final Model Comparison ────────────────────────
model_names_5 = ['Ridge', 'Lasso', 'Random Forest']
cv_rmse_5     = [ridge_rmse, lasso_rmse, rf_rmse]
train_rmse_5  = [
    float(np.sqrt(mean_squared_error(y, ridge.predict(X_train)))),
    float(np.sqrt(mean_squared_error(y, lasso.predict(X_train)))),
    float(np.sqrt(mean_squared_error(y, rf.predict(X_train)))),
]
if HAVE_XGB:
    model_names_5.append('XGBoost')
    cv_rmse_5.append(xgb_rmse)
    train_rmse_5.append(float(np.sqrt(mean_squared_error(y, xgb.predict(X_train)))))

comparison = pd.DataFrame({
    'Model':       model_names_5,
    'CV RMSE':     cv_rmse_5,
    'Train RMSE':  train_rmse_5,
})
comparison['Overfit Gap'] = comparison['CV RMSE'] - comparison['Train RMSE']
comparison = comparison.sort_values('CV RMSE').reset_index(drop=True)
comparison.index += 1

print("📊 Bảng xếp hạng (theo CV RMSE):")
print(comparison.to_string())

# ── 2 biểu đồ cạnh nhau ───────────────────────────────────
s_models = comparison['Model'].tolist()
s_cv      = comparison['CV RMSE'].tolist()
s_train   = comparison['Train RMSE'].tolist()
best_cv   = min(s_cv)
dot_colors = ['#27ae60' if r == best_cv else '#3498db' for r in s_cv]

fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(s_models) + 1)))

# ── Left: Lollipop ────────────────────────────────────────
y_pos = range(len(s_models))
for i, (cv_v, tr_v, col) in enumerate(zip(s_cv, s_train, dot_colors)):
    axes[0].plot([tr_v, cv_v], [i, i], color='#bdc3c7', lw=2, zorder=1)
    axes[0].scatter(cv_v, i, s=200, c=col,       zorder=4, edgecolors='white', lw=1.5)
    axes[0].scatter(tr_v, i, s=80,  c='#ecf0f1', zorder=3, edgecolors='#7f8c8d', lw=1.2, marker='D')
    axes[0].text(cv_v + 0.0008, i, f'{cv_v:.4f}', va='center', fontsize=9, fontweight='bold')
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(s_models, fontsize=11)
axes[0].set_xlabel('RMSE', fontsize=10)
axes[0].set_title('🟢 CV RMSE  ◆ Train RMSE\n(khoảng cách = mức độ overfit)',
                   fontsize=11, fontweight='bold')
axes[0].axvline(best_cv, color='#27ae60', linestyle=':', alpha=0.5,
                label=f'Best = {best_cv:.4f}')
axes[0].legend(fontsize=9)
axes[0].set_xlim(min(s_train) * 0.96, max(s_cv) * 1.04)
axes[0].grid(True, axis='x', alpha=0.3)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# ── Right: Train vs CV scatter (overfit view) ─────────────
axes[1].scatter(s_train, s_cv, s=200, c=dot_colors, edgecolors='white', lw=1.5, zorder=3)
for i, nm in enumerate(s_models):
    axes[1].annotate(nm, (s_train[i], s_cv[i]),
                     textcoords='offset points', xytext=(6, 4), fontsize=9)
lim_min = min(min(s_train), min(s_cv)) * 0.97
lim_max = max(max(s_train), max(s_cv)) * 1.02
axes[1].plot([lim_min, lim_max], [lim_min, lim_max], 'r--', lw=1.5, alpha=0.7,
             label='Train = CV (không overfit)')
axes[1].set_xlabel('Train RMSE', fontsize=10)
axes[1].set_ylabel('CV RMSE', fontsize=10)
axes[1].set_title('Train vs CV RMSE — Overfitting Map\n(phía trên đường đỏ = đang overfit)',
                   fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle('Task 5 — Final Model Comparison: Performance & Overfitting',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

best_model_final = comparison.iloc[0]
print(f"\n🥇 Model tốt nhất: {best_model_final['Model']}")
print(f"   CV RMSE = {best_model_final['CV RMSE']:.4f}")
print(f"   → Sẽ dùng model này để tạo file submission ở Buổi 8!")


**Expected Output:**

![img7-10](images/img_buoi7/img7-10.png)
![img7-11](images/img_buoi7/img7-11.png)

---

## 🎯 Chiến Lược Giảm Sai Số — Chọn Cách Nào?

Sau khi có kết quả đánh giá, câu hỏi thực tế là: **làm sao để RMSE xuống thấp hơn?**  
Có 2 hướng chính, mỗi hướng phù hợp với mục tiêu khác nhau:

| | **S1 — Chính Xác Tối Đa** | **S2 — Đơn Giản & Ổn Định** |
|---|---|---|
| **Kỹ thuật** | XGBoost + Stacking + Feature Engineering nâng cao | Ridge / Lasso blend cơ bản |
| **RMSE mục tiêu** | ~0.107 (Top 10% Kaggle) | ~0.113 (Top 30%) |
| **Phải chịu** | Phức tạp, tốn thời gian train, khó giải thích | Không đạt RMSE tối đa |
| **Phù hợp** | Fintech, ngân hàng, proptech lớn | Startup, học tập, deploy nhanh |

> **Không có chiến lược nào sai** — tuỳ mục tiêu mà chọn.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

current_best  = min(s_cv)
kaggle_target = 0.115
kaggle_top10  = 0.100

s_rmse   = [current_best * 0.96, current_best * 1.02]
S1_COLOR = '#e63946'
S2_COLOR = '#2a9d8f'

criteria  = ['Độ chính xác', 'Dễ deploy', 'Chi phí thấp', 'Dễ giải thích', 'Tốc độ train']
scores_s1 = [5, 1, 1, 2, 1]
scores_s2 = [3, 5, 5, 4, 5]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#f8f9fa')
for ax in (ax1, ax2):
    ax.set_facecolor('white')
    ax.spines[['top', 'right']].set_visible(False)

# ── Panel 1: RMSE ─────────────────────────────────────────────────────────────
y_pos = [1, 0]
bars  = ax1.barh(y_pos, s_rmse, height=0.42,
                  color=[S1_COLOR, S2_COLOR],
                  edgecolor='white', linewidth=1.5, alpha=0.88, zorder=3)

for bar, val, col in zip(bars, s_rmse, [S1_COLOR, S2_COLOR]):
    ax1.text(val + 0.00015, bar.get_y() + bar.get_height() / 2,
             f'{val:.4f}', va='center', ha='left',
             color=col, fontsize=11, fontweight='bold')

ax1.axvline(current_best,  color='#f4a261', lw=2, ls='--', zorder=4)
ax1.axvline(kaggle_target, color='#2a9d8f', lw=2, ls='-.', zorder=4)
ax1.axvline(kaggle_top10,  color='#e63946', lw=2, ls=':',  zorder=4)
ax1.axvspan(kaggle_top10, kaggle_target, alpha=0.06, color='#2a9d8f')

# Nhãn đường tham chiếu đặt trên — không đè bar
for xv, col, lbl in [
    (kaggle_top10,  '#e63946', f'Top 10%\n{kaggle_top10}'),
    (current_best,  '#f4a261', f'Hiện tại\n{current_best:.4f}'),
    (kaggle_target, '#2a9d8f', f'Top 30%\n{kaggle_target}'),
]:
    ax1.text(xv, 1.72, lbl, ha='center', va='bottom', color=col,
             fontsize=7.5, fontweight='bold',
             bbox=dict(fc='white', ec=col, lw=0.8, pad=2, boxstyle='round,pad=0.25'))

ax1.set_yticks(y_pos)
ax1.set_yticklabels(['S1: Chính Xác Tối Đa', 'S2: Đơn Giản & Ổn Định'],
                     fontsize=10.5, fontweight='bold')
ax1.set_xlabel('RMSE (thấp hơn = tốt hơn)', fontsize=9, color='#555')
ax1.set_title('RMSE Mục Tiêu', fontsize=12, fontweight='bold', pad=38)
ax1.set_xlim(kaggle_top10 * 0.987, max(s_rmse) * 1.055)
ax1.set_ylim(-0.55, 2.3)
ax1.grid(True, axis='x', alpha=0.15, ls='--')

# ── Panel 2: Tiêu chí ─────────────────────────────────────────────────────────
x     = np.arange(len(criteria))
width = 0.32

b1 = ax2.bar(x - width / 2, scores_s1, width, color=S1_COLOR, alpha=0.85,
              label='S1: Chính Xác Tối Đa', edgecolor='white', linewidth=1, zorder=3)
b2 = ax2.bar(x + width / 2, scores_s2, width, color=S2_COLOR, alpha=0.85,
              label='S2: Đơn Giản & Ổn Định', edgecolor='white', linewidth=1, zorder=3)

for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width() / 2, h + 0.1,
             f'{int(h)}/5', ha='center', va='bottom',
             fontsize=8.5, fontweight='bold', color=bar.get_facecolor())

ax2.set_xticks(x)
ax2.set_xticklabels(criteria, fontsize=9)
ax2.set_yticks([1, 2, 3, 4, 5])
ax2.set_yticklabels(['1', '2', '3', '4', '5'], fontsize=8, color='#888')
ax2.set_ylim(0, 7)
ax2.set_title('So Sánh Tiêu Chí', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9, loc='upper center', bbox_to_anchor=(0.5, -0.16),
           ncol=2, frameon=True, framealpha=0.9, edgecolor='#ccc')
ax2.grid(True, axis='y', alpha=0.15, ls='--')

fig.suptitle('2 Chiến Lược Giảm RMSE — Dự Đoán Giá Nhà', fontsize=13, fontweight='bold', y=1.0)
plt.tight_layout()
plt.show()


**Expected Output:**

![img7-12](images/img_buoi7/img7-12.png)

### 🔴 S1 — Chính Xác Tối Đa

- **Kỹ thuật:** XGBoost + Stacking + Feature Engineering nâng cao
- **RMSE mục tiêu:** ~0.106 (Top 10% Kaggle)
- **Ưu điểm:** RMSE thấp nhất, cạnh tranh được top leaderboard
- **Nhược điểm:** Phức tạp, tốn thời gian train, khó giải thích cho khách hàng
- **Phù hợp:** Fintech · Ngân hàng đầu tư · Proptech lớn
- **Chọn khi:** Accuracy là ưu tiên số 1, team đủ mạnh và có budget

---

### 🟢 S2 — Đơn Giản & Ổn Định ★ Khuyên dùng

- **Kỹ thuật:** Ridge / Lasso blend cơ bản
- **RMSE mục tiêu:** ~0.113 (Top 30% Kaggle)
- **Ưu điểm:** Dễ deploy, chi phí thấp, giải thích được cho khách hàng
- **Nhược điểm:** RMSE cao hơn S1 khoảng 0.004–0.007
- **Phù hợp:** Startup BĐS · App định giá · SME · Học tập
- **Chọn khi:** Cần deploy nhanh, team nhỏ, hoặc cần giải thích rõ ràng

> 💡 **Gợi ý thực tế:** Bắt đầu với **S2** để có baseline chạy được trước, sau đó nâng dần lên **S1** nếu kết quả chưa đủ tốt.


---

## 📝 Kiểm Tra Nhanh — Ôn Lại Kiến Thức Buổi 7

*Làm trong 5 phút — điền vào chỗ trống hoặc chọn 1 đáp án đúng.*

---

**Câu 1 (Điền từ):** Residual của một điểm dữ liệu được định nghĩa là:

Residual đo sai số của '_____'. Nếu tổng tất cả residuals ≈ 0 → model không có xu hướng đoán quá cao hoặc quá thấp.

---

**Câu 2 (Trắc nghiệm):** Khi model có **Train RMSE rất thấp** nhưng **CV RMSE cao hơn nhiều**, vấn đề là:

- A) Underfit — model quá đơn giản
- B) Overfit — model học vẹt training data
- C) Dữ liệu bị mất cân bằng
- D) Số features quá ít

---

**Câu 3 (Điền từ):** Learning Curve vẽ RMSE theo ___ (trục X) khi tăng dần từ 20% đến 100%.

---

**Câu 4 (Đúng/Sai):** Q-Q Plot lý tưởng sẽ có tất cả các điểm nằm **gần đường thẳng đỏ**.

> *(Đúng / Sai?)*

---

**Câu 5 (Trắc nghiệm):** Khi chọn model để submit Kaggle, ta nên dùng chỉ số nào?

- A) Train R²
- B) Train RMSE
- C) CV RMSE
- D) MAE trên training data

---

**Câu 6 (Điền từ):** Model có CV RMSE: `mean=0.115`, `std=0.020`. Độ lệch này được đánh giá là ___ (ổn định / không ổn định).

> *(Gợi ý: std > 0.015 là cảnh báo)*

---

> 🔍 Xem đáp án ở ô markdown bên dưới!


In [ ]:
# ── Minh Họa: CV fold + Residuals — Ridge vs XGBoost ──────
# (Đây là visualization minh họa, không phải bài tập code)

# 1. CV fold comparison
ridge_folds = -cross_val_score(ridge, X_train, y,
                                scoring='neg_root_mean_squared_error', cv=kf)
if HAVE_XGB:
    xgb_folds = -cross_val_score(xgb, X_train, y,
                                  scoring='neg_root_mean_squared_error', cv=kf)

fold_labels = [f'Fold {i}' for i in range(1, 6)]
x = np.arange(5)
w = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Panel 1: Grouped bar — fold by fold
bars1 = axes[0].bar(x - w/2, ridge_folds, w, label='Ridge',
                    color='#3498db', edgecolor='none', alpha=0.85)
if HAVE_XGB:
    bars2 = axes[0].bar(x + w/2, xgb_folds, w, label='XGBoost',
                        color='#e67e22', edgecolor='none', alpha=0.85)
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0002,
                 f'{bar.get_height():.4f}', ha='center', fontsize=7)
if HAVE_XGB:
    for bar in bars2:
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0002,
                     f'{bar.get_height():.4f}', ha='center', fontsize=7)
axes[0].set_xticks(x)
axes[0].set_xticklabels(fold_labels)
axes[0].set_ylabel('RMSE')
axes[0].set_title('CV RMSE từng Fold — Ridge vs XGBoost\n(độ ổn định giữa các fold)',
                   fontweight='bold')
axes[0].legend()
axes[0].grid(True, axis='y', alpha=0.3)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Panel 2: Residuals scatter — Ridge vs XGBoost
res_ridge = y - ridge.predict(X_train)
if HAVE_XGB:
    res_xgb = y - xgb.predict(X_train)
    pred_xgb = xgb.predict(X_train)

pred_ridge = ridge.predict(X_train)
axes[1].scatter(pred_ridge, res_ridge, alpha=0.25, s=12, color='#3498db', label='Ridge')
if HAVE_XGB:
    axes[1].scatter(pred_xgb, res_xgb, alpha=0.25, s=12, color='#e67e22', label='XGBoost')
axes[1].axhline(0, color='red', linestyle='--', lw=1.5)
axes[1].set_xlabel('Predicted (log SalePrice)')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residuals: Ridge vs XGBoost\n(XGBoost khít hơn trên train data)',
                   fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.2)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle('So Sánh Ridge vs XGBoost — Stability & Residuals', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

##Gợi ý: in ra std và mean của CV folds để so sánh độ ổn định giữa Ridge và XGBoost


**Expected Output:**

![img7-13](images/img_buoi7/img7-13.png)

## ✅ Đáp Án & Lời Giải

**Câu 1:** $\text{Residual} = y_{\text{thực}} - \hat{y}_{\text{dự đoán}}$
- Residual đo sai số của **từng điểm**. Nếu tổng tất cả residuals ≈ 0 → model không có xu hướng đoán quá cao hoặc quá thấp.

---

**Câu 2:** **B — Overfit**
- Train RMSE thấp = model **thuộc lòng** training data, bao gồm cả nhiễu (noise).
- CV RMSE cao = model **không generalize** tốt trên data mới.
- Giải pháp: thêm regularization, giảm complexity, hoặc thu thập thêm data.

---

**Câu 3:** **Số lượng training samples** (train_sizes)
- Learning curve tăng dần kích thước training set từ 20% → 100%, quan sát xem Train và Validation RMSE hội tụ như thế nào.

---

**Câu 4:** **Đúng**
- Q-Q Plot so sánh quantiles của residuals với phân phối chuẩn lý thuyết.
- Điểm sát đường đỏ = residuals phân phối chuẩn = model đủ tốt, không có outlier nghiêm trọng.

---

**Câu 5:** **C — CV RMSE**
- Đây là ước tính **trung thực nhất** cho performance trên data mới (Kaggle test set).
- Train RMSE luôn thấp hơn thực tế vì model đã "nhìn thấy" data đó rồi.

---

**Câu 6:** **Không ổn định** (std = 0.020 > ngưỡng cảnh báo 0.015)
- Độ lệch cao → RMSE dao động nhiều giữa các fold → model nhạy cảm với cách chia data.
- Giải pháp: thêm regularization, dùng nhiều fold hơn (10-fold), hoặc thu thập thêm data.


---

## 🏁 Tổng Kết Buổi 7

| Task | Nội dung | Kỹ năng đạt được |
|------|---------|-----------------|
| **Task 1 — Metrics** | RMSE, MAE, R² + biểu đồ | Đọc và chọn metric phù hợp với bài toán |
| **Task 2 — CV Fold** | Bar chart + Box plot ổn định | Phát hiện model dễ bị ảnh hưởng bởi data |
| **Task 3 — Learning Curves** | Train vs Val RMSE theo train_size | Chẩn đoán overfit / underfit |
| **Task 4 — Residuals** | Scatter + Histogram + Q-Q plot | Tìm điểm model còn thiếu sót |
| **Task 5 — So sánh** | Lollipop + Overfit scatter | Chọn model tốt nhất để submit |
| **Optimization** | 4 chiến lược + 4 biểu đồ | Biết bước tiếp theo để cải thiện |

---

## 🗺️ Nhìn Lại Toàn Bộ Pipeline: 7 Buổi Đã Đi Qua

| Buổi | Chủ đề | Output chính |
|------|--------|-------------|
| **1** | Giới thiệu & Mục tiêu | Hiểu bài toán, 79 features, lộ trình 9 buổi |
| **2** | Setup & EDA cơ bản | SalePrice lệch → cần `log1p()` |
| **3** | EDA nâng cao | Biết features quan trọng nhất (tương quan) |
| **4** | Data Cleaning | Data sạch, xử lý NaN + outliers |
| **5** | Feature Engineering | +TotalSF, HouseAge, log/poly, encode, scale |
| **6** | Model Training | 5 models với CV RMSE — chọn được best model |
| **7** | Evaluation & Chiến Lược | Metrics toàn diện + lộ trình tối ưu tiếp theo |

---

### 🔄 Pipeline Tổng Quát

```
📥 Raw Data (train.csv, test.csv)
         │
         ▼
[Buổi 3–4] Data Cleaning
  • Fill NaN theo logic từng cột
  • Xóa outliers cực đoan
  • Target: y = log1p(SalePrice)
         │
         ▼
[Buổi 5] Feature Engineering
  • Tạo features tổng hợp: TotalSF, TotalBath, HouseAge…
  • Ordinal Encoding + One-Hot Encoding
  • Variance Filter + StandardScaler
         │
         ▼
[Buổi 6] Model Training         [Buổi 7] Evaluation
  Ridge / Lasso               → RMSE, MAE, R²
  Random Forest               → CV fold stability
  XGBoost                     → Learning Curves
  Stacking                    → Residual Analysis
         │
         ▼
[Buổi 8] Kaggle Submission → [Buổi 9] Quiz Tốt Nghiệp
```

---

### ➡️ Bước Tiếp Theo (Buổi 8)

- Dùng model tốt nhất để **tạo file submission** cho Kaggle
- Chuyển đổi ngược: `SalePrice = expm1(y_pred)` để ra giá thực tế
- Lưu file `submission.csv` đúng format Kaggle
